# 01 — Haversine Distance (The Actually Correct Way)

The previous notebook computed a distance in degrees and called it a day. This one fixes that.

The **Haversine formula** computes the great-circle distance between two points on a sphere — the shortest path along the surface of the Earth. The result is in kilometers. It is consistent regardless of latitude and does not require pretending the planet is flat.

## 1. The Problem Recap

From the previous two notebooks:

- **Earth ≠ flat.** Latitude and longitude are angular measurements on a sphere.
- **Degrees ≠ distance.** One degree of longitude covers ~111 km at the equator and ~0 km at the poles. The Euclidean formula treats both axes as uniform, so its output in degrees is not a real distance.

What we need is a formula that:
1. Works in spherical geometry, not flat geometry
2. Produces output in actual length units (kilometers)
3. Gives consistent results regardless of where on Earth the points are

That formula is Haversine.

## 2. The Haversine Formula

The formula computes the central angle between two points on a sphere, then multiplies by the Earth's radius to get arc length.

```text
a = sin²((φ₂ − φ₁) / 2) + cos(φ₁) · cos(φ₂) · sin²((λ₂ − λ₁) / 2)

d = 2 · r · arcsin(√a)
```

Where:
- `φ` = latitude in radians
- `λ` = longitude in radians
- `r` = Earth's mean radius ≈ **6371 km**
- `d` = great-circle distance in kilometers

You do not need to derive this. You need to understand what each part does:

| Part | What it handles |
|------|----------------|
| `sin²((φ₂ − φ₁) / 2)` | North-south angular separation |
| `cos(φ₁) · cos(φ₂) · sin²((λ₂ − λ₁) / 2)` | East-west angular separation, scaled by latitude |
| `arcsin(√a)` | Converts combined angle to arc |
| `2 · r · ...` | Converts arc to surface distance |

The `cos(φ₁) · cos(φ₂)` term is what makes this correct. It is the longitude correction that the Euclidean formula omits — it shrinks the east-west contribution as latitude increases, matching how meridians actually converge toward the poles.

## 3. Variables Before Code

Before implementing, get the variable names clear:

```text
Input (GeoJSON order):
    p1 = [lon1, lat1]
    p2 = [lon2, lat2]

Convert to radians:
    φ₁ = math.radians(lat1)
    φ₂ = math.radians(lat2)
    λ₁ = math.radians(lon1)
    λ₂ = math.radians(lon2)

Differences:
    Δφ = φ₂ − φ₁    ← latitude difference in radians
    Δλ = λ₂ − λ₁    ← longitude difference in radians

Earth radius:
    R = 6371.0  km
```

The only conversion step is degrees → radians. Everything after that is plugging into the formula.

## 4. Implementation in Python

Build it step by step first, then collapse it into a clean function.

In [3]:
import math

# Same two points as the previous notebook — [lon, lat]
p1 = (-98.5, 33.8)
p2 = (-97.2, 34.1)

R = 6371.0  # Earth's mean radius in km

# Step 1 — convert to radians
lat1, lon1 = math.radians(p1[1]), math.radians(p1[0])
lat2, lon2 = math.radians(p2[1]), math.radians(p2[0])

# Step 2 — differences
d_lat = lat2 - lat1
d_lon = lon2 - lon1

# Step 3 — Haversine intermediate value
a = math.sin(d_lat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(d_lon / 2)**2

# Step 4 — central angle → surface distance
d = 2 * R * math.asin(math.sqrt(a))

print(f"Haversine distance: {d:.2f} km")

Haversine distance: 124.46 km


In [4]:
# Clean reusable version
def haversine_km(p1, p2):
    """
    Great-circle distance between two [lon, lat] points in kilometers.
    """
    R = 6371.0
    lat1, lon1 = math.radians(p1[1]), math.radians(p1[0])
    lat2, lon2 = math.radians(p2[1]), math.radians(p2[0])
    d_lat = lat2 - lat1
    d_lon = lon2 - lon1
    a = math.sin(d_lat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(d_lon / 2)**2
    return 2 * R * math.asin(math.sqrt(a))


print(f"{haversine_km(p1, p2):.2f} km")

124.46 km


## 5. Compare to Euclidean

Same two points. Same operation. Very different outputs.

In [5]:
def euclidean_distance(p1, p2):
    return math.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)


pairs = [
    ("Wichita Falls → nearby",      (-98.5, 33.8),   (-97.2, 34.1)),
    ("Wichita Falls → OKC",         (-98.5, 33.8),   (-97.5, 35.5)),
    ("Dallas → Houston",            (-96.80, 32.78), (-95.37, 29.76)),
    ("Dallas → New York",           (-96.80, 32.78), (-74.01, 40.71)),
    ("Dallas → London",             (-96.80, 32.78), ( -0.13, 51.51)),
]

print(f"{'Pair':<30}  {'Euclidean (°)':>14}  {'Haversine (km)':>15}")
print("-" * 64)
for label, a, b in pairs:
    e = euclidean_distance(a, b)
    h = haversine_km(a, b)
    print(f"{label:<30}  {e:>13.4f}°  {h:>14.1f} km")

Pair                             Euclidean (°)   Haversine (km)
----------------------------------------------------------------
Wichita Falls → nearby                 1.3342°           124.5 km
Wichita Falls → OKC                    1.9723°           210.0 km
Dallas → Houston                       3.3415°           362.3 km
Dallas → New York                     24.1303°          2205.4 km
Dallas → London                       98.4678°          7640.8 km


The Euclidean column is unitless noise. The Haversine column is something you can read on a map, verify with a ruler, and act on. That is the difference worth caring about.

## 6. Visualize on a Map

Draw the same line as before. This time, annotate it with a real distance.

In [6]:
from ipyleaflet import Map, GeoJSON

p1 = (-98.5, 33.8)
p2 = (-97.2, 34.1)
dist_km = haversine_km(p1, p2)

points_fc = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": list(p1)}, "properties": {"name": "p1"}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": list(p2)}, "properties": {"name": "p2"}},
    ]
}

line_fc = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "geometry": {"type": "LineString", "coordinates": [list(p1), list(p2)]},
        "properties": {"distance_km": round(dist_km, 2)}
    }]
}

print(f"Distance: {dist_km:.2f} km")

center_lat = (p1[1] + p2[1]) / 2
center_lon = (p1[0] + p2[0]) / 2

m = Map(center=(center_lat, center_lon), zoom=9)
m.add(GeoJSON(data=points_fc))
m.add(GeoJSON(data=line_fc, style={"color": "#2a9d8f", "weight": 2}))
m

Distance: 124.46 km


Map(center=[33.95, -97.85], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

---

## Exercise A — Distances Between Cities

Compute the Haversine distance between each pair of cities below. Print the results in a readable table.

In [7]:
city_pairs = [
    ("Wichita Falls → Dallas",     (-98.49, 33.91), (-96.80, 32.78)),
    ("Dallas → San Antonio",       (-96.80, 32.78), (-98.49, 29.42)),
    ("OKC → Tulsa",                (-97.52, 35.47), (-95.99, 36.15)),
    ("Lubbock → Amarillo",         (-101.87, 33.57), (-101.83, 35.22)),
    ("El Paso → Dallas",           (-106.49, 31.76), (-96.80, 32.78)),
]

def haversine_km(p1, p2):
    """
    Great-circle distance between two [lon, lat] points in kilometers.
    """
    R = 6371.0
    lat1, lon1 = math.radians(p1[1]), math.radians(p1[0])
    lat2, lon2 = math.radians(p2[1]), math.radians(p2[0])
    d_lat = lat2 - lat1
    d_lon = lon2 - lon1
    a = math.sin(d_lat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(d_lon / 2)**2
    return 2 * R * math.asin(math.sqrt(a))

print(f"{'Route':<30} {'Distance (km)':>14}")
print("-" * 46)
for label, p1, p2 in city_pairs:
    dist = haversine_km(p1, p2)
    print(f"{label:<30} {dist:>14.2f}")

# your code here
# print each pair name and Haversine distance in km

Route                           Distance (km)
----------------------------------------------
Wichita Falls → Dallas                 201.07
Dallas → San Antonio                   406.77
OKC → Tulsa                            157.33
Lubbock → Amarillo                     183.51
El Paso → Dallas                       917.75


## Exercise B — Nearest Airfield by Real Distance

Using `haversine_km`, find the nearest airfield to the base point. Compare your answer to the Euclidean ranking from the previous notebook — does the order change?

In [8]:
base = (-98.47, 33.91)

airfields = [
    ("Tinker AFB",         (-97.37, 35.39)),
    ("NAS Fort Worth JRB", (-97.04, 32.85)),
    ("Lubbock",            (-101.87, 33.57)),
    ("Oklahoma City",      (-97.52, 35.47)),
    ("Abilene",            (-99.73, 32.45)),
]

ranked = sorted(
    airfields,
    key = lambda item: haversine_km(base, item[1])
)

print(f"{'Rank':<5} {'Airfield':<22} {'Haversine (km)':>15}")
print("-" * 58)
for rank, (name, coord) in enumerate(ranked, 1):
    hav  = haversine_km(base, coord)
    print(f"{rank:<5} {name:<22} {hav:>15.2f} ")




Rank  Airfield                Haversine (km)
----------------------------------------------------------
1     NAS Fort Worth JRB              177.54 
2     Tinker AFB                      192.89 
3     Oklahoma City                   193.99 
4     Abilene                         200.26 
5     Lubbock                         316.63 


## Exercise C — Sanity Check Your Results

A good habit: after computing distances, verify at least one result against an external reference (Google Maps, a known fact, etc.).

Pick any one city pair from Exercise A. Look up the driving distance or straight-line distance using any outside source. How close is your Haversine result? If it differs, explain why (hint: driving distance follows roads; Haversine is straight-line on a sphere).

In [10]:
# your pair, your verification, your note on any difference
pair_name = "Dallas → San Antonio"
haversine_result = haversine_km((-96.80, 32.78), (-98.49, 29.42))
external_reference = 410.0   # replace with the km value you looked up

print(f"Haversine: {haversine_result:.1f} km")
if external_reference:
    diff = abs(haversine_result - external_reference)
    print(f"External:  {external_reference} km")
    print(f"Difference: {diff:.1f} km")
    print("Note: ...")

Haversine: 406.8 km
External:  410.0 km
Difference: 3.2 km
Note: ...


---

## Check Your Understanding

A student copies the Haversine implementation but accidentally passes coordinates in `[lat, lon]` order instead of `[lon, lat]`:

```python
# Intended: Dallas [lon=-96.80, lat=32.78], Houston [lon=-95.37, lat=29.76]
result = haversine_km(
    (32.78, -96.80),   # [lat, lon] — wrong order
    (29.76, -95.37)
)
print(result)
```

**Question:** Will this crash, silently produce a wrong answer, or produce the correct answer? Explain what the function actually receives and how that affects the output.

```python
# your answer here
```


---

## Next

In [02 — Compare Methods](./02-Compare_Methods.ipynb), we put Euclidean and Haversine side by side to make the error growth visible — finding the crossover point where the flat-Earth approximation stops being acceptable.